# 01 — Exploratory analysis: AI4I 2020 machine failure data

Ten thousand production rows from a milling process. Each row is one tool pass
with five sensor channels and a `Machine failure` flag, plus five one-hot
columns naming *which* failure mode fired.

This notebook only establishes the facts the modelling decisions rest on:
how rare failures are, how uneven the five modes are, how strongly the two
temperatures and the two mechanical channels move together, and where in the
speed/torque plane the fatal combinations sit. Nothing here trains a model.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/ai4i2020.csv", encoding="utf-8-sig")  # the CSV ships with a BOM
print(df.shape)
df.head()

In [ ]:
print(df.dtypes)
print("missing values:", int(df.isna().sum().sum()))

## 1. The target is rare

3.39% positives means accuracy is useless: predicting "never fails" scores 96.6%.
Every number downstream is ROC-AUC (ranking), average precision (ranking at the
rare end) or precision/recall at a chosen operating point.

In [ ]:
print(df["Machine failure"].value_counts())
print(f"failure rate: {100 * df['Machine failure'].mean():.2f}%")

## 2. Five failure modes, very different sizes

The mode flags are the reason to think about sub-problems: each mode has its own
physical driver, so a model that is good on average can still miss a whole mode.

In [ ]:
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
print(df[modes].sum().to_string())
print("rows with more than one mode:", int((df[modes].sum(axis=1) > 1).sum()))

## 3. The label itself is not perfectly consistent

`Machine failure` is supposed to be the OR of the five mode flags. It is not:
a handful of rows are flagged with no mode attached, and more rows carry a mode
but a zero overall flag. This bounds how good any model can look on the overall
target, and it is worth stating rather than silently averaging over.

In [ ]:
any_mode = (df[modes].sum(axis=1) > 0).astype(int)
print("flag=1 but no mode set :", int(((df["Machine failure"] == 1) & (any_mode == 0)).sum()))
print("mode set but flag=0    :", int(((df["Machine failure"] == 0) & (any_mode == 1)).sum()))

## 4. Operating class matters

`Type` is the product quality variant. Failure rate is not constant across it,
which is why it is tempting as a feature — see the ablation in the README for
what it actually does to test AUC.

In [ ]:
df.groupby("Type")["Machine failure"].agg(["size", "sum", "mean"]).round(4)

## 5. Two channels are near-redundant, two are nearly independent

Air and process temperature are almost the same signal (r ≈ 0.88), and torque is
strongly inverse to rotational speed (r ≈ -0.88) because the machine holds power
roughly constant. Trees tolerate this, but it explains why SHAP and permutation
importance disagree about which of torque/speed is "the" driver: correlated
features split credit between them.

In [ ]:
cols = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]",
        "Torque [Nm]", "Tool wear [min]"]
print(df[cols].corr().round(3).to_string())

## 6. Where in the envelope failures happen

Power failures cluster at low speed with high torque; overstrain failures at high
speed with lower torque. Heat-dissipation failures sit in the high-torque band
regardless of speed. A single tree ensemble can carve those regions out, which is
the practical argument for a tree model over a linear one on five raw channels.

In [ ]:
for mode in ["PWF", "OSF", "HDF", "TWF"]:
    sub = df[df[mode] == 1]
    print(f"{mode}: n={len(sub):3d}  rpm median={sub['Rotational speed [rpm]'].median():6.0f}"
          f"  torque median={sub['Torque [Nm]'].median():5.1f}"
          f"  tool wear median={sub['Tool wear [min]'].median():5.0f}")

## 7. Tool wear is a slow clock, not a switch

Tool wear runs 0–253 min and failures are spread across it rather than triggered
at a threshold — except for TWF, which is by construction a wear-driven label but
still not separable on these five channels alone. That is the single clearest
limitation of the tabular setup: a genuine sequence model would see the wear ramp.

In [ ]:
df.groupby("Machine failure")["Tool wear [min]"].describe().round(1)

## Takeaways carried into the pipeline

1. 3.39% positives → stratified split, `class_weight="balanced"`, rank metrics.
2. Five heterogeneous modes → report per-mode detection, not just overall recall.
3. Labelling noise (~27 rows) → don't chase the last decimal of AUC.
4. Correlated channels → check feature attribution two ways (SHAP + permutation).
5. No timestamps, no machine ID → this is a static tabular task; sequence modelling
   would need data this file does not contain.